# Bronze ingestion with Auto Loader (dynamic)

Loads every folder in the `source` container into its own bronze table.

| Widget | Meaning |
|---|---|
| `folder_name` | One folder to load, e.g. `orders`. Leave **empty** to load **all** folders. |
| `exclude` | Comma-separated folders to skip, e.g. `regions`. |
| `file_format` | File format of the source files (`parquet`, `csv`, `json`). |
| `target_schema` | Catalog and schema the tables are written to. |

Each folder gets its own table, schema folder and checkpoint, so each run only loads new files.

In [0]:
dbutils.widgets.text("folder_name", "", "Folder (empty = all)")
dbutils.widgets.text("exclude", "", "Folders to skip (comma-separated)")
dbutils.widgets.text("file_format", "parquet", "File format")
dbutils.widgets.text("target_schema", "databricks_catalog.bronze_schema", "Target schema")

In [0]:
SOURCE_ROOT = "abfss://source@db4ete.dfs.core.windows.net"
STATE_ROOT = "abfss://bronze@db4ete.dfs.core.windows.net"   # schemas + checkpoints live here

folder_name = dbutils.widgets.get("folder_name").strip()
exclude = {f.strip() for f in dbutils.widgets.get("exclude").split(",") if f.strip()}
file_format = dbutils.widgets.get("file_format").strip()
target_schema = dbutils.widgets.get("target_schema").strip()

# Decide which folders to process
if folder_name:
    folders = [folder_name]
else:
    folders = [
        f.name.rstrip("/")
        for f in dbutils.fs.ls(SOURCE_ROOT + "/")
        if f.isDir() and not f.name.startswith(("_", "."))
    ]

folders = [f for f in folders if f not in exclude]
print(f"Folders to process: {folders}")

Folders to process: ['customers', 'orders', 'products']


In [0]:
def ingest(folder: str) -> int:
    """Load new files from source/<folder>/ into <target_schema>.<folder>. Returns rows added."""
    table = f"{target_schema}.{folder}"

    query = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", file_format)
            .option("cloudFiles.schemaLocation", f"{STATE_ROOT}/_schemas/{folder}")
            .load(f"{SOURCE_ROOT}/{folder}/")
        .writeStream
            .option("checkpointLocation", f"{STATE_ROOT}/_checkpoints/{folder}")
            .trigger(availableNow=True)
            .toTable(table)
    )
    query.awaitTermination()

    # Serverless may return progress as objects instead of dicts, so handle both
    rows = 0
    for p in (query.recentProgress or []):
        value = p.get("numInputRows") if isinstance(p, dict) else getattr(p, "numInputRows", None)
        rows += value or 0   # some progress entries report None instead of 0
    return rows

In [0]:
results, failures = {}, {}

for folder in folders:
    print(f"▶ {folder} ...")
    try:
        results[folder] = ingest(folder)
        print(f"  ✓ {results[folder]} new rows → {target_schema}.{folder}")
    except Exception as e:
        failures[folder] = str(e)
        print(f"  ✗ failed: {type(e).__name__}: {e}")

print("\nSummary")
for folder, rows in results.items():
    print(f"  {folder}: {rows} new rows")

if failures:
    raise Exception(f"Ingestion failed for: {list(failures)}")

▶ customers ...
  ✓ 0 new rows → databricks_catalog.bronze_schema.customers
▶ orders ...
  ✓ 0 new rows → databricks_catalog.bronze_schema.orders
▶ products ...
  ✓ 0 new rows → databricks_catalog.bronze_schema.products

Summary
  customers: 0 new rows
  orders: 0 new rows
  products: 0 new rows


In [0]:
%sql
SHOW TABLES IN databricks_catalog.bronze_schema;

database,tableName,isTemporary
bronze_schema,customers,false
bronze_schema,orders,false
bronze_schema,products,false
bronze_schema,regions,false
